# 实验五 · 单算子调用 —— aclnn 两段式接口

**所属**：《并行计算》第七章 · AscendCL 应用开发　|　**难度**：⭐⭐⭐ 进阶　|　**预计时长**：40~50 分钟

CANN 的算子库中已有数千个经过深度优化的内置算子，可以直接调用。调用这些算子的接口分为两段：第一段在主机侧完成入参校验、Shape 推导与切块（Tiling），第二段才把核函数下发到 Stream。

两段的划分不只是接口形式上的划分，**两段各自的耗时可以分别测量**。本实验把一次算子调用的耗时分为三项：主机执行第一段 `aclnnXxxGetWorkspaceSize` 的时间、主机执行第二段 `aclnnXxx` 的时间、设备端执行该算子的时间，并逐项测出它与数据量的关系。

本实验的另一部分内容是 `aclTensor`。它用**视图**与**存储**两套形状描述同一块内存，因此转置与切片可以在不复制数据的前提下表达出来。

> **实验说明**
> 1. 本实验的核心内容有三点：两段式接口各自做什么、workspace 的生命周期、`aclTensor` 的视图与存储双描述。
> 2. 本实验全程只调用 `aclnnAdd` 一个算子。这是有意的安排：接口的形式与调用的开销与算子的具体功能无关，换成其它算子，本实验的结论同样成立。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。
> 4. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**。
> 5. 本实验**不需要任何来自其它实验的读数**，两组测量的判据全部由本次运行的数据给出。


## 🎯 学习目标

完成本实验后，学生应能够：

- 说出两段式接口中每一段各自完成哪些工作，并指出哪一段在主机侧、哪一段是异步下发
- 说明 `aclOpExecutor` 由第一段产生、被第二段消耗，因而**每一次调用都必须重新执行第一段**
- 按三条规则管理 workspace：申请与释放前用 `if (workspaceSize > 0)` 判断、在同步之后释放、同一条 Stream 上的多个算子可共用一块
- 说出 `aclCreateTensor` 九个参数的三组含义，并写出行优先连续张量的 strides
- 用视图、strides 与 offset 三者，在不复制数据的前提下表达转置与切片
- 按正确顺序销毁资源：先描述符后数据内存，并说明反过来的后果
- 说出 CANN 算子的四个类别与对应的库文件，并解释 `libopapi.so` 为什么被拆开
- 由实测判断固定开销在主机侧与设备侧各有多大，以及它在什么规模上会成为主导
- 说明用原子算子组合出一个复合运算时，固定开销为什么会随调用次数成倍增加


## 🗺️ 学习路径

1. **接口范式**：认识两段式接口的分工，以及 executor 与 workspace 各自的生命周期
2. **数据描述**：`aclTensor` 用视图与存储两套形状描述同一块内存，由此理解销毁的顺序
3. **工程约束**：算子分四类、库文件也分四个，链接哪些库由用到哪类算子决定
4. **组合开销**：复合运算需要由多个原子算子组合而成，每增加一次调用就增加一份固定开销
5. **程序实现**：把两段式调用写成可运行的程序，并让同一份代码支持两种测量模式
6. **拆解耗时**：把一次调用的耗时分成主机执行第一段、主机执行第二段与设备执行三项，逐项由实测给出它与数据量的关系
7. **零拷贝视图**：在同一块存储上建立三种视图，验证转置与切片确实不必复制数据


## 1. 背景与动机：算子从哪里来

开发一个算子与调用一个已有的算子，是两个不同的问题，在工程中所占的比重也不相同：**绝大多数应用并不开发算子，而是调用已有的算子并将它们组合起来**。本实验讨论的是后一个问题。

CANN 的算子库按用途分成四类，覆盖了从基础数学运算到大模型核心结构的常见需求。调用它们的接口统一以 `aclnn` 为前缀，采用**两段式**形式：

```cpp
aclnnStatus aclnnXxxGetWorkspaceSize(const aclTensor *src, ..., aclTensor *out, ...,
                                     uint64_t *workspaceSize, aclOpExecutor **executor);
aclnnStatus aclnnXxx(void *workspace, uint64_t workspaceSize,
                     aclOpExecutor *executor, aclrtStream stream);
```

### 1.1 本实验要回答的三个问题

**第一个问题：一次算子调用的耗时由哪几项组成。** 两段式接口的两段各自在主机侧占用一段时间：第一段 `aclnnXxxGetWorkspaceSize` 做校验、Shape 推导与 Tiling，第二段 `aclnnXxx` 把核函数放进 Stream；此外设备端执行这个算子也要占用一段时间。§11 把这三项分开测量，并让数据量跨四个数量级变化，**逐项判断它是否随数据量增长**。

**第二个问题是规模。** 一次算子调用的耗时中，有一部分与数据量无关，称为**固定开销**。数据量减小到何种规模时，固定开销将超过设备端的实际计算时间？这一规模可以由实测确定。

**第三个问题是描述方式。** `aclTensor` 为什么要同时给出视图形状与存储形状？§12 在同一块存储上建立三种视图，验证转置与切片能否在不复制数据的前提下表达出来。

### 1.2 本实验不测什么

- **不测算子库的覆盖范围。** 有哪些算子可用请查《算子库》中的「算子接口（aclnn）」，本实验只用 `Add` 一个。
- **不测自定义算子的接入。** 本实验只调用 CANN 内置的算子；自己开发的算子如何被应用调用，不在本实验范围内。
- **不做算子级的性能剖析。** 内置算子内部如何切块、使用了哪些片上缓冲，本实验均不涉及，可观察到的只有它的入参与耗时。
- **不比较内置算子与手写实现的性能。** 本实验讨论的是**调用一个算子需要付出多少开销**，而不是某个算子的运算速度。


## 2. 两段式接口

### 2.1 两段各做什么

<img src="images/07.05_aclnn_flow.png" alt="单算子 API 调用流程" width="460">

单算子 API 调用流程

流程图中虚线框内的六个步骤构成一次单算子调用的全部内容。其中计算 workspace 大小并申请内存、执行算子这两步，对应的正是两段式接口的两段。

图例用两种颜色区分六个步骤：只有传输数据一步是绿色的可选步骤，若数据已经位于设备上（例如上一个算子的输出直接作为本算子的输入），该步骤可以省略；其余五步均为必选。§12 的三个视图共用同一块已经位于设备上的存储，因此不涉及任何数据传输。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
|  | 第一段 `aclnnXxxGetWorkspaceSize` | 第二段 `aclnnXxx` |
| --- | --- | --- |
| 执行位置 | **主机侧** | 主机侧下发，设备侧执行 |
| 做的事 | 入参校验、动态 Shape 下推导输出 Shape、**数据切块（Tiling）**、计算所需 workspace 大小 | executor 执行计算，框架自动调用 DFX（Dump、溢出检测）与 Runtime 的 LaunchKernel |
| 产出 | `workspaceSize` 与 `executor` | 无返回值以外的产出，结果写在 `out` 张量里 |
| 是否阻塞 | 同步，返回时工作已完成 | **异步**，返回时核函数可能尚未开始 |
| 与数据量的关系 | 见 §11 的实测 | 见 §11 的实测 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"></th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">第一段 <code>aclnnXxxGetWorkspaceSize</code></th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">第二段 <code>aclnnXxx</code></th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">执行位置</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>主机侧</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">主机侧下发，设备侧执行</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">做的事</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">入参校验、动态 Shape 下推导输出 Shape、<strong>数据切块（Tiling）</strong>、计算所需 workspace 大小</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">executor 执行计算，框架自动调用 DFX（Dump、溢出检测）与 Runtime 的 LaunchKernel</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">产出</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>workspaceSize</code> 与 <code>executor</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">无返回值以外的产出，结果写在 <code>out</code> 张量里</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">是否阻塞</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同步，返回时工作已完成</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>异步</strong>，返回时核函数可能尚未开始</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与数据量的关系</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">见 §11 的实测</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">见 §11 的实测</td>
</tr>
</tbody>
</table>

第二段是异步的：**返回不等于完成，取结果之前必须同步。**

### 2.2 一个完整的最小调用

下面这段代码是一次完整的两段式调用，只保留了必要的语句，未做错误检查。§6 要编译运行的探针程序即以此为基本结构：

```cpp
uint64_t workspaceSize = 0;
aclOpExecutor* executor = nullptr;

// 第一段：主机侧的准备工作，不下发任何任务
aclnnAddGetWorkspaceSize(self, other, alpha, out, &workspaceSize, &executor);

// 按第一段算出的大小申请 workspace
void* workspace = nullptr;
if (workspaceSize > 0) {
  aclrtMalloc(&workspace, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST);
}

// 第二段：把核函数下发到 stream
aclnnAdd(workspace, workspaceSize, executor, stream);

// 异步下发，必须同步之后才能取结果
aclrtSynchronizeStream(stream);
```

### 2.3 executor 的生命周期

`aclOpExecutor` 由第一段产生，被第二段消耗。**它不能留到下一次调用。** 后续涉及多次调用 aclnnAdd 算子，则需调用多次第一段接口，获取不同的 aclOpExecutor。

这一条决定了本实验的核心测量内容。既然每一次调用都要重新执行第一段，**第一段的耗时就不是一次性的初始化开销，而是每次调用都要付出的固定开销**。它的具体数值由 §11 测出。

> ⚠️ 一个直接的推论：**不要试图缓存 executor 来加速循环。** 《应用开发指南》要求多次调用同一算子时逐次调用第一段以获取不同的 executor。

### 2.4 workspace 的三条规则

workspace 是算子执行期间使用的临时设备内存，大小由第一段的 Tiling 结果决定。管理它有三条规则：

1. **`workspaceSize` 可能为 0，申请与释放两处都要用 `if (workspaceSize > 0)` 检查。**
2. **释放必须在 `aclrtSynchronizeStream` 之后。** 第二段是异步的，返回时核函数可能还没开始执行；此时释放 workspace，核函数将读写已经释放的内存。
3. **同一条 Stream 上顺序执行的多个算子可以共用一块 workspace**，容量取各自需求的最大值。同一条 Stream 上的任务严格按下发顺序执行，前一个算子使用完毕之后后一个才开始，因此不存在竞争。**并发运行在不同 Stream 上的算子不能共用**：这些算子可能同时执行，共用一块临时内存会造成相互覆盖。

## 3. aclTensor：视图与存储

### 3.1 九个参数分成三组

`aclCreateTensor` 的参数数量较多，但可以按含义分为三组：

```cpp
aclTensor *aclCreateTensor(const int64_t *viewDims, uint64_t viewDimsNum,   // ① 视图形状
                           aclDataType dataType,                            //   数据类型
                           const int64_t *stride, int64_t offset,           // ② 视图到存储的映射
                           aclFormat format,
                           const int64_t *storageDims, uint64_t storageDimsNum,  // ③ 存储形状
                           void *addr);                                      //   存储首地址
```

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 组 | 参数 | 含义 |
| --- | --- | --- |
| ① 视图 | `viewDims` / `viewDimsNum` | **算子按什么形状**读写这个张量 |
| ② 映射 | `stride` / `offset` | 视图中第 $(i,j,\dots)$ 个元素对应存储中的第几个元素 |
| ③ 存储 | `storageDims` / `storageDimsNum` / `addr` | 这块内存**实际的形状**，以及它的首地址 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">组</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">参数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">① 视图</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>viewDims</code> / <code>viewDimsNum</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>算子按什么形状</strong>读写这个张量</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">② 映射</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>stride</code> / <code>offset</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">视图中第 $(i,j,\dots)$ 个元素对应存储中的第几个元素</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">③ 存储</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>storageDims</code> / <code>storageDimsNum</code> / <code>addr</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">这块内存<strong>实际的形状</strong>，以及它的首地址</td>
</tr>
</tbody>
</table>

把算子读写的形状与内存的实际布局分开描述，由此得到的好处是**切片、转置与广播都可以不复制数据**：只改变第 ① 组与第 ② 组参数，第 ③ 组保持不变。§12 将在同一块存储上建立三个视图并分别做加法，验证这一点。

### 3.2 strides 以元素为单位

$$\text{strides}[i] = \text{shape}[i+1] \times \text{strides}[i+1], \qquad \text{strides}[\text{rank}-1] = 1$$

两点容易出错：

- **单位是元素，不是字节。** 一个 $4 \times 8$ 的 float 张量，`strides` 是 `{8, 1}` 而不是 `{32, 4}`。
- **按从右向左的次序递推。** 最右一维的 stride 恒为 1（行优先连续存放），其余各维由其右侧一维的形状与 stride 相乘得到。

本实验的 `ContiguousStrides` 写成一个循环：

```cpp
std::vector<int64_t> strides(shape.size(), 1);
for (int64_t i = shape.size() - 2; i >= 0; i--) {
  strides[i] = shape[i + 1] * strides[i + 1];
}
```

### 3.3 销毁顺序

**先销毁描述符（`aclDestroyTensor` / `aclDestroyScalar`），再释放数据内存（`aclrtFree`）。**

若次序颠倒，`aclrtFree` 已经释放了内存，而张量描述符仍然持有指向这块内存的地址，随后的 `aclDestroyTensor` 将在一个已经失效的地址上执行。这类错误通常不会立即报错，而是**在其后的某次内存分配中表现为难以解释的失败**。

## 4. 算子的分类与库文件

CANN 的算子按用途分为四类，**每一类对应一个库文件**：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 类别 | 内容 | 例子 | 需要链接的库 |
| --- | --- | --- | --- |
| Math 类 | 数值计算：张量形态变换、基础数学运算、随机数生成 | `Add`、`Abs`、`Muls` | `libopapi_math.so` |
| NN 类 | 深度学习模型中的常见计算 | 卷积、矩阵乘、激活函数、归一化 | `libopapi_math.so` 与 `libopapi_nn.so` |
| CV 类 | 图像处理与目标检测 | `GridSample` | `libopapi_math.so` 与 `libopapi_cv.so` |
| Transformer 类 | 大模型核心算子 | Attention 类、LayerNorm 类、通算融合类 | `libopapi_math.so` 与 `libopapi_transformer.so` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">类别</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内容</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">例子</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">需要链接的库</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Math 类</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数值计算：张量形态变换、基础数学运算、随机数生成</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Add</code>、<code>Abs</code>、<code>Muls</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>libopapi_math.so</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">NN 类</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">深度学习模型中的常见计算</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">卷积、矩阵乘、激活函数、归一化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>libopapi_math.so</code> 与 <code>libopapi_nn.so</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CV 类</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">图像处理与目标检测</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>GridSample</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>libopapi_math.so</code> 与 <code>libopapi_cv.so</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Transformer 类</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">大模型核心算子</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Attention 类、LayerNorm 类、通算融合类</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>libopapi_math.so</code> 与 <code>libopapi_transformer.so</code></td>
</tr>
</tbody>
</table>

**`libopapi_math.so` 是四类的共同前提**，其余三个库各自附加。本实验只用到 `Add`，它是 Math 类算子，因此只需链接 `libopapi_math.so`；**若要调用 `Matmul` 这类 NN 类算子，还须再加上 `libopapi_nn.so`**——按这张表决定链接参数，是每一个调用内置算子的应用都要做的一件事。

> ⚠️ **从 CANN 8.5.0 起，全量算子总库 `libopapi.so` 已废弃，请勿使用。**

头文件同样可以按类引用总头文件 `aclnnop/aclnn_ops_math.h`、`aclnnop/aclnn_ops_nn.h` 等，也可以只引用单个算子的头文件 `aclnnop/aclnn_*.h`（`*` 表示具体算子名）。本实验用后者，因为只用到一个算子。

> ⚠️ **单算子头文件的名字与算子名并非总是一一对应。** 例如 `aclnnMuls` 声明在 `aclnnop/aclnn_mul.h` 中，而不是 `aclnn_muls.h`。不确定时引用该类的总头文件（如 Math 类的 `aclnn_ops_math.h`），或到 `${ASCEND_HOME_PATH}/include/aclnnop/` 下按算子名搜一次。

库文件这一侧还有一条与算子分类无关、但同样影响链接的规则：Runtime 库有两个名字，`libacl_rt.so` 是现行推荐，`libascendcl.so` 是为兼容旧版本保留、后续版本会废弃。

## 5. 原子算子的组合

`aclnn` 提供的是**原子算子**，一个接口完成一项运算。由此带来一个直接的后果：**任何复合运算都需要由若干次调用组合而成，每一次调用都要完整执行一遍两段式接口。**

以矩阵乘的常见形式 $C = \alpha AB + \beta C$ 为例，`aclnn` 中没有一个接口能够一次完成该运算，需要将它分解为四步：

$$\text{Matmul}: P = AB \quad\to\quad \text{Muls}: S_1 = \alpha P \quad\to\quad \text{Muls}: S_2 = \beta C \quad\to\quad \text{Add}: O = S_1 + S_2$$

**代价是固定开销增加到四倍。** §11 将测出一次调用中第一段在主机侧所需的时间，该时间在一次 $C = \alpha AB + \beta C$ 中要付出四次，且与数据规模无关。数据规模越小，这部分开销所占的比例越大。减少这部分开销的途径只有一条：**减少调用次数**，即在《算子库》中查找能够一次完成其中若干步的融合算子。

**收益有两条。** 其一，算子库不必为每一种组合各提供一个接口：$\alpha$、$\beta$ 的缩放与加法都是通用运算，不必与矩阵乘绑定。其二，组合方式由调用者决定，若某个模型只需要计算 $C = AB$，就不必付出后三步的开销。

> 💡 组合时中间结果可以复用同一块设备内存，依据与 §2.4 第 3 条相同：**这几个算子在同一条 Stream 上顺序执行**，前一个使用完毕之后后一个才开始。

本实验的程序只调用一个 `aclnnAdd`，这样 §11 测得的三项耗时可以准确对应到一次调用上。组合所带来的开销由本节说明，不再另行编写更复杂的程序加以演示。


## 6. 环境准备与检查

先创建源码目录，再把 CANN 的环境变量导入当前内核。这两步与前四个实验相同。


In [ ]:
!mkdir -p src_aclnn

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")


下面这一格检查编译器与三个库文件：Runtime 库、算子公共数据类型库与 **Math 类算子库**。本实验只调用 `Add`，它属于 Math 类（§4）。

Runtime 库有两个名字：`libacl_rt.so` 是现行推荐，`libascendcl.so` 是为兼容旧版本保留、后续版本会废弃，因此按这一顺序查找，取第一个存在的库。


In [ ]:
import os, shutil

ascend_home = os.environ.get("ASCEND_HOME_PATH", "")
lib_dirs = [
    path
    for path in (f"{ascend_home}/lib64", f"{ascend_home}/devlib")
    if os.path.isdir(path)
]


# 按优先级在库目录中查找库，返回第一个存在的库名（不含 lib 前缀与 .so 后缀）
def find_lib(candidates):
    for name in candidates:
        for lib_dir in lib_dirs:
            if os.path.exists(os.path.join(lib_dir, f"lib{name}.so")):
                return name
    return None


print("ASCEND_HOME_PATH :", ascend_home or "⚠️  未设置")
print("g++              :", shutil.which("g++") or "⚠️  未找到")

selected = []
for purpose, candidates in [
    ("Runtime", ["acl_rt", "ascendcl"]),
    ("算子公共数据类型", ["nnopbase"]),
    # libopapi.so 自 CANN 8.5.0 起废弃（§4），不作为回退项：
    # 若这一项返回 None，说明算子包未按类拆分安装，请按 §4 检查安装。
    ("Math 类算子", ["opapi_math"]),
]:
    name = find_lib(candidates)
    print(f"{purpose:<18} 候选 {candidates} -> {name}")
    if name is not None:
        selected.append(name)

ACL_LIBDIRS = ["-L" + path for path in lib_dirs]
ACL_LIBS = ["-l" + name for name in selected]
print("链接参数         :", " ".join(ACL_LIBS))
print()
print(
    "✅ 环境就绪，可以开始实验。"
    if ascend_home and shutil.which("g++") and len(selected) == 3
    else "⚠️  环境不完整，请检查上面的输出。"
)


接下来运行一次最小的两段式调用。这段程序只处理八个元素，是在 §2.2 的基本结构上补全而成的可运行程序，用来确认四件事：**算子库链接正确、第一段能够算出 workspace 大小、第二段的计算结果正确，以及 `aclrtMalloc` 申请 0 字节在本机上的返回码**。它同时也是 §3.3 所述销毁顺序的第一个实例：第 5 步先销毁描述符，再释放数据内存。


In [ ]:
%%writefile src_aclnn/aclnn_probe.cpp
// A complete two-phase call, small enough to read in one screen. It confirms
// that the operator libraries are linked correctly and prints the workspace
// that the first phase asks for.
#include <cstdint>
#include <cstdio>

#include "acl/acl.h"
#include "aclnnop/aclnn_add.h"

int main() {
  if (aclInit(nullptr) != ACL_SUCCESS || aclrtSetDevice(0) != ACL_SUCCESS) {
    std::printf("[PROBE] aclnn_ok=0 reason=init\n");
    return 1;
  }
  aclrtStream stream = nullptr;
  if (aclrtCreateStream(&stream) != ACL_SUCCESS) {
    std::printf("[PROBE] aclnn_ok=0 reason=stream\n");
    return 1;
  }

  // 1. Device memory for the two inputs and the output.
  const int64_t shape[1] = {8};
  const int64_t strides[1] = {1};
  const size_t bytes = 8 * sizeof(float);
  float host[8] = {0.0f, 1.0f, 2.0f, 3.0f, 4.0f, 5.0f, 6.0f, 7.0f};
  void* self_addr = nullptr;
  void* other_addr = nullptr;
  void* out_addr = nullptr;
  aclrtMalloc(&self_addr, bytes, ACL_MEM_MALLOC_HUGE_FIRST);
  aclrtMalloc(&other_addr, bytes, ACL_MEM_MALLOC_HUGE_FIRST);
  aclrtMalloc(&out_addr, bytes, ACL_MEM_MALLOC_HUGE_FIRST);
  aclrtMemcpy(self_addr, bytes, host, bytes, ACL_MEMCPY_HOST_TO_DEVICE);
  aclrtMemcpy(other_addr, bytes, host, bytes, ACL_MEMCPY_HOST_TO_DEVICE);

  // 2. Tensors and the scalar. Arguments 1-2 are the view shape, 4-5 map the
  //    view onto the storage, and 7-8 are the storage shape.
  aclTensor* self = aclCreateTensor(shape, 1, ACL_FLOAT, strides, 0,
                                    ACL_FORMAT_ND, shape, 1, self_addr);
  aclTensor* other = aclCreateTensor(shape, 1, ACL_FLOAT, strides, 0,
                                     ACL_FORMAT_ND, shape, 1, other_addr);
  aclTensor* out = aclCreateTensor(shape, 1, ACL_FLOAT, strides, 0,
                                   ACL_FORMAT_ND, shape, 1, out_addr);
  float alpha_value = 1.0f;
  aclScalar* alpha = aclCreateScalar(&alpha_value, ACL_FLOAT);
  if (self == nullptr || other == nullptr || out == nullptr ||
      alpha == nullptr) {
    std::printf("[PROBE] aclnn_ok=0 reason=create\n");
    return 1;
  }

  // 3. First phase on the host, then the workspace, then the second phase.
  uint64_t workspace_size = 0;
  aclOpExecutor* executor = nullptr;
  const int first = aclnnAddGetWorkspaceSize(self, other, alpha, out,
                                             &workspace_size, &executor);
  if (first != ACL_SUCCESS) {
    std::printf("[PROBE] aclnn_ok=0 reason=phase1 code=%d\n", first);
    return 1;
  }
  void* workspace = nullptr;
  if (workspace_size > 0) {
    aclrtMalloc(&workspace, workspace_size, ACL_MEM_MALLOC_HUGE_FIRST);
  }
  const int second = aclnnAdd(workspace, workspace_size, executor, stream);
  if (second != ACL_SUCCESS) {
    std::printf("[PROBE] aclnn_ok=0 reason=phase2 code=%d\n", second);
    return 1;
  }
  aclrtSynchronizeStream(stream);

  // 4. Read the result back only after the stream has finished.
  float result[8] = {0.0f};
  aclrtMemcpy(result, bytes, out_addr, bytes, ACL_MEMCPY_DEVICE_TO_HOST);
  std::printf("[PROBE] aclnn_ok=1 workspace=%llu out0=%.1f out7=%.1f\n",
              static_cast<unsigned long long>(workspace_size),
              static_cast<double>(result[0]), static_cast<double>(result[7]));

  // 4b. What does aclrtMalloc return for a request of zero bytes? The guard of
  //     section 2.4 is taken from the official sample; whether the call would
  //     actually fail without it is measured here rather than asserted.
  void* zero_probe = nullptr;
  const int zero_ret =
      aclrtMalloc(&zero_probe, 0, ACL_MEM_MALLOC_HUGE_FIRST);
  std::printf("[PROBE] malloc_zero_ret=%d ptr=%s\n", zero_ret,
              (zero_probe == nullptr) ? "null" : "non-null");
  if (zero_probe != nullptr) {
    aclrtFree(zero_probe);
  }

  // 5. Descriptors first, then the buffers they describe.
  aclDestroyScalar(alpha);
  aclDestroyTensor(out);
  aclDestroyTensor(other);
  aclDestroyTensor(self);
  if (workspace != nullptr) {
    aclrtFree(workspace);
  }
  aclrtFree(out_addr);
  aclrtFree(other_addr);
  aclrtFree(self_addr);
  aclrtDestroyStream(stream);
  aclrtResetDevice(0);
  aclFinalize();
  return 0;
}


编译探针所需的库与主程序相同：Runtime 库、算子公共数据类型库与 Math 类算子库。


In [ ]:
import os, subprocess

cmd = (
    ["g++", "src_aclnn/aclnn_probe.cpp", "-std=c++17", "-O2", "-Wall"]
    + ["-I" + os.environ["ASCEND_HOME_PATH"] + "/include"]
    + ACL_LIBDIRS
    + ACL_LIBS
    + ["-o", "src_aclnn/aclnn_probe"]
)
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print((proc.stdout + proc.stderr).strip())
else:
    print(
        subprocess.run(
            ["./src_aclnn/aclnn_probe"], capture_output=True, text=True
        ).stdout
    )


## 7. 本实验的测量设计

### 7.1 两组测量

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 模式 | 回答的问题 | 输出记录行 |
| --- | --- | --- |
| `phase` | 两段各自在主机侧占用多少时间？这部分耗时是否与数据量有关？设备端执行该算子需要多少时间？ | `[PHASE]` |
| `view` | 改变视图、strides 与 offset，能否在不复制数据的前提下表达转置与切片？ | `[VIEW]` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">模式</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">回答的问题</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">输出记录行</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>phase</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两段各自在主机侧占用多少时间？这部分耗时是否与数据量有关？设备端执行该算子需要多少时间？</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>[PHASE]</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>view</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">改变视图、strides 与 offset，能否在不复制数据的前提下表达转置与切片？</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>[VIEW]</code></td>
</tr>
</tbody>
</table>

### 7.2 参数的选择

**`phase` 的元素数取 $2^{10}$ 到 $2^{24}$，每档翻四倍，共八档。** 下界 1024 个元素（4 KB）已经小到设备端几乎没有实际计算量，上界 $2^{24}$ 个元素恰好是 64 MB。**跨越四个数量级**是这一组参数的关键：只有让数据量的变化范围足够大，才能分辨出哪几项随数据量增长、哪几项保持不变。

**每一档预热 2 次、测量 10 次取平均。** 预热用来排除首次触碰设备内存与首次加载算子的一次性开销。

**`view` 的规模取 512。** 这一组只做正确性判定，不测耗时，规模只需大到能够划分出一个四分之一子块即可。三个视图的结果各自与主机侧算出的期望值比较，逐元素最大绝对误差不超过 $10^{-5}$ 即判定通过。


## 8. 程序实现

程序分五段写入 `src_aclnn/acl_aclnn.cpp`。第一段用 `%%writefile`（覆盖），其余各段用 `%%writefile -a`（追加）。

### 8.1 头文件、常量与错误检查

本段只引入 `aclnn_add.h` 一个算子头文件。常量 `kRelTolerance` 即 §7.2 为 `view` 一组规定的容差。

In [ ]:
%%writefile src_aclnn/acl_aclnn.cpp
/**
 * Parallel Computing, Chapter 7: Calling a Built-in Operator
 *
 * The aclnn interface is split into two phases. The first phase runs entirely
 * on the host: it validates the arguments, infers the output shape, performs
 * tiling and reports how much workspace the kernel needs. The second phase
 * submits the kernel to a stream. This program measures what each phase
 * costs and what the workspace depends on, and it checks that a view, its
 * strides and its offset can express a transpose or a slice without copying.
 *
 * Usage: acl_aclnn <mode> [arguments]
 *   phase <repeat>   cost of each phase against the element count
 *   view  <n>        zero-copy views over one storage buffer
 */
#include <cstdint>  // int64_t, uint64_t
#include <cstdio>   // std::printf, std::fprintf
#include <cstdlib>  // std::atoi, std::strtoll
#include <cstring>  // std::strcmp
#include <ctime>    // clock_gettime, timespec
#include <vector>   // std::vector

#include "acl/acl.h"            // Runtime resource management APIs
#include "aclnnop/aclnn_add.h"  // Single-operator API of Add

namespace {

constexpr int32_t kDeviceId = 0;
constexpr int kMatDims = 2;
constexpr int kWarmupRuns = 2;
constexpr int kMaxExecutors = 32;
constexpr float kOneValue = 1.0f;
constexpr double kRelTolerance = 1.0e-5;

}  // namespace

// Checks the return code of an acl API. On failure it prints the API name,
// the return code and the error message, then returns immediately.
#define ACL_CHECK(expr)                                                     \
  do {                                                                      \
    const int acl_ret = static_cast<int>(expr);                             \
    if (acl_ret != ACL_SUCCESS) {                                           \
      const char* err_msg = aclGetRecentErrMsg();                           \
      std::fprintf(stderr, "[ERR] api=%s code=%d msg=%s\n", #expr, acl_ret, \
                   (err_msg == nullptr) ? "(no message)" : err_msg);        \
      return acl_ret;                                                       \
    }                                                                       \
  } while (0)


### 8.2 主机侧的四个工具函数

这一段不涉及 AscendCL 接口，四个函数都是后续反复使用的通用工具：

- `GetTimeMs` 用 `clock_gettime(CLOCK_MONOTONIC)` 读取主机侧的时刻。它自系统启动起单调递增，不受校时影响，**测量时间间隔必须用它**；
- `FillMatrix` 按一个取值有界且不重复的规律填充矩阵，为 `view` 一组构造输入数据；
- `ContiguousStrides` 按行优先次序从右向左计算 strides，对应 §3.1 的第 4–5 个参数，**单位是元素而不是字节**；
- `MaxAbsDiff` 返回两块主机缓冲区之间逐元素绝对差的最大值，作为 `view` 一组的判定依据。


In [ ]:
%%writefile -a src_aclnn/acl_aclnn.cpp
// Returns a monotonic timestamp in milliseconds, for the host-side clock.
double GetTimeMs() {
  timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1.0e6;
}

// Fills a row-major matrix with a bounded, non-periodic pattern. The two
// operands take different moduli so that no row of A equals any column of B.
void FillMatrix(float* data, int64_t rows, int64_t cols, int64_t modulus) {
  for (int64_t i = 0; i < rows; ++i) {
    for (int64_t j = 0; j < cols; ++j) {
      data[i * cols + j] = static_cast<float>((i + j) % modulus) * 0.001f;
    }
  }
}

// Computes row-major strides in elements, right to left. The unit is the
// element rather than the byte, and the rightmost stride is always 1.
void ContiguousStrides(const int64_t* shape, int rank, int64_t* strides) {
  strides[rank - 1] = 1;
  for (int i = rank - 2; i >= 0; --i) {
    strides[i] = shape[i + 1] * strides[i + 1];
  }
}

// Returns the largest absolute difference between two host buffers.
double MaxAbsDiff(const float* lhs, const float* rhs, int64_t count) {
  double worst = 0.0;
  for (int64_t i = 0; i < count; ++i) {
    const double diff =
        static_cast<double>(lhs[i]) - static_cast<double>(rhs[i]);
    const double magnitude = (diff < 0.0) ? -diff : diff;
    if (magnitude > worst) {
      worst = magnitude;
    }
  }
  return worst;
}


### 8.3 张量的创建与销毁

`MakeTensor` 将 §3.1 的九个参数封装在一处，调用时只需给出形状、数据类型与设备地址；`CreateDeviceTensor` 在此基础上一并申请对应的设备内存；`DestroyDeviceTensor` 固定了 §3.3 规定的销毁次序，即**先销毁描述符，再释放数据内存**，程序中所有释放操作都经由该函数完成。


In [ ]:
%%writefile -a src_aclnn/acl_aclnn.cpp
// One device buffer together with the tensor that describes it.
struct DeviceTensor {
  void* addr;
  aclTensor* tensor;
};

// Creates a contiguous tensor over a buffer that already exists. The nine
// arguments of aclCreateTensor fall into three groups: 1-2 are the view shape,
// 4-5 map the view onto the storage, and 7-8 are the storage shape.
aclTensor* MakeTensor(const int64_t* shape, int rank, aclDataType dtype,
                      void* addr) {
  int64_t strides[4] = {0, 0, 0, 0};
  ContiguousStrides(shape, rank, strides);
  return aclCreateTensor(shape, static_cast<uint64_t>(rank), dtype, strides, 0,
                         ACL_FORMAT_ND, shape, static_cast<uint64_t>(rank),
                         addr);
}

// Allocates a device buffer and creates a contiguous tensor over it. The view
// shape and the storage shape are identical here, which is the ordinary case.
int CreateDeviceTensor(const int64_t* shape, int rank, aclDataType dtype,
                       DeviceTensor* out) {
  int64_t count = 1;
  for (int i = 0; i < rank; ++i) {
    count *= shape[i];
  }
  const size_t width =
      (dtype == ACL_FLOAT16) ? sizeof(uint16_t) : sizeof(float);
  out->addr = nullptr;
  out->tensor = nullptr;
  ACL_CHECK(aclrtMalloc(&out->addr, static_cast<size_t>(count) * width,
                        ACL_MEM_MALLOC_HUGE_FIRST));
  out->tensor = MakeTensor(shape, rank, dtype, out->addr);
  if (out->tensor == nullptr) {
    std::fprintf(stderr,
                 "[ERR] api=aclCreateTensor code=- msg=returned null\n");
    return ACL_ERROR_INVALID_PARAM;
  }
  return ACL_SUCCESS;
}

// Releases a tensor. The descriptor goes first and the data buffer second:
// the reverse order would leave the descriptor pointing at freed memory.
void DestroyDeviceTensor(DeviceTensor* item) {
  if (item->tensor != nullptr) {
    (void)aclDestroyTensor(item->tensor);
    item->tensor = nullptr;
  }
  if (item->addr != nullptr) {
    (void)aclrtFree(item->addr);
    item->addr = nullptr;
  }
}


### 8.4 两段各自的耗时

`RunPhase` 对每一档元素数进行两遍测量，两遍所回答的问题不同：

- **第一遍**在同一个循环内分别记录两段的起止时刻，得到的是调用一次算子时主机在每一段上花费的时间；
- **第二遍**先连续执行 `runs` 次第一段，再连续下发 `runs` 次第二段，并在下发的前后各记录一个 Event。两个 Event 之间的区间不包含主机侧的 Tiling，得到的是设备端连续完成 `runs` 次该算子所需的时间。

> ⚠️ **第二遍测得的区间包含哪些内容。** 该区间虽然不含第一段，但仍然包含每一次第二段的下发时间：主机将任务提交到队列的这段时间里，设备处于空闲状态。因此 §11 由这一列读出的设备侧下限，是下发时间与启动时间之和的下限，而不是核函数启动时间本身。数据规模越小，下发时间在该区间中所占的比例越大。

两遍之所以分开进行，是因为第一遍的循环中主机与设备交替工作，此时由 Event 界定的区间会将主机侧的时间一并计入，所得数值既不表示主机耗时，也不表示设备耗时。


In [ ]:
%%writefile -a src_aclnn/acl_aclnn.cpp
// Grows the workspace buffer when an operator asks for more than is held.
// Freeing the old buffer means giving memory back that a kernel already
// submitted may still be reading, so the stream is synchronised first. This is
// rule 2 of section 2.4 applied to the workspace itself.
int EnsureWorkspace(void** buffer, uint64_t* capacity, uint64_t needed,
                    aclrtStream stream) {
  if (needed <= *capacity) {
    return ACL_SUCCESS;
  }
  if (*buffer != nullptr) {
    ACL_CHECK(aclrtSynchronizeStream(stream));
    ACL_CHECK(aclrtFree(*buffer));
    *buffer = nullptr;
  }
  ACL_CHECK(aclrtMalloc(buffer, static_cast<size_t>(needed),
                        ACL_MEM_MALLOC_HUGE_FIRST));
  *capacity = needed;
  return ACL_SUCCESS;
}

// Measures the host-side cost of each phase, and the device-side duration of
// the same operator, against the number of elements. One Add is used
// throughout so that only the amount of data changes between rows.
int RunPhase(int repeat) {
  // 2^10 to 2^24 elements, four times per step: four orders of magnitude of
  // data, which is what makes a data-independent cost visible as a flat line.
  const int64_t counts[] = {1LL << 10, 1LL << 12, 1LL << 14, 1LL << 16,
                            1LL << 18, 1LL << 20, 1LL << 22, 1LL << 24};
  const int kNumCounts = static_cast<int>(sizeof(counts) / sizeof(counts[0]));
  const int64_t widest = counts[kNumCounts - 1];
  const int runs = (repeat < kMaxExecutors) ? repeat : kMaxExecutors;

  aclrtStream stream = nullptr;
  ACL_CHECK(aclrtCreateStream(&stream));
  aclrtEvent begin = nullptr;
  aclrtEvent end = nullptr;
  ACL_CHECK(aclrtCreateEventExWithFlag(&begin, ACL_EVENT_TIME_LINE));
  ACL_CHECK(aclrtCreateEventExWithFlag(&end, ACL_EVENT_TIME_LINE));

  void* buffer[3] = {nullptr, nullptr, nullptr};
  for (int i = 0; i < 3; ++i) {
    const size_t bytes = static_cast<size_t>(widest) * sizeof(float);
    ACL_CHECK(aclrtMalloc(&buffer[i], bytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMemset(buffer[i], bytes, 0, bytes));
  }
  const float one = kOneValue;
  aclScalar* scalar = aclCreateScalar(const_cast<float*>(&one), ACL_FLOAT);
  if (scalar == nullptr) {
    std::fprintf(stderr,
                 "[ERR] api=aclCreateScalar code=- msg=returned null\n");
    return ACL_ERROR_INVALID_PARAM;
  }

  void* workspace = nullptr;
  uint64_t capacity = 0;
  std::printf("[CFG] mode=phase repeat=%d points=%d\n", runs, kNumCounts);

  for (int index = 0; index < kNumCounts; ++index) {
    const int64_t shape[1] = {counts[index]};
    aclTensor* self = MakeTensor(shape, 1, ACL_FLOAT, buffer[0]);
    aclTensor* other = MakeTensor(shape, 1, ACL_FLOAT, buffer[1]);
    aclTensor* out = MakeTensor(shape, 1, ACL_FLOAT, buffer[2]);
    if (self == nullptr || other == nullptr || out == nullptr) {
      std::fprintf(stderr, "[ERR] api=MakeTensor code=- msg=returned null\n");
      return ACL_ERROR_INVALID_PARAM;
    }

    uint64_t needed = 0;
    aclOpExecutor* executor = nullptr;
    for (int warm = 0; warm < kWarmupRuns; ++warm) {
      ACL_CHECK(aclnnAddGetWorkspaceSize(self, other, scalar, out, &needed,
                                         &executor));
      ACL_CHECK(EnsureWorkspace(&workspace, &capacity, needed, stream));
      ACL_CHECK(aclnnAdd(workspace, needed, executor, stream));
    }
    ACL_CHECK(aclrtSynchronizeStream(stream));

    // First pass: the host-side cost of each phase, measured in place.
    double first_ms = 0.0;
    double second_ms = 0.0;
    for (int run = 0; run < runs; ++run) {
      const double t0 = GetTimeMs();
      ACL_CHECK(aclnnAddGetWorkspaceSize(self, other, scalar, out, &needed,
                                         &executor));
      const double t1 = GetTimeMs();
      ACL_CHECK(aclnnAdd(workspace, needed, executor, stream));
      const double t2 = GetTimeMs();
      first_ms += t1 - t0;
      second_ms += t2 - t1;
    }
    ACL_CHECK(aclrtSynchronizeStream(stream));

    // Second pass: all first-phase calls are made up front so that the device
    // interval covers the kernels and their submission, and nothing else.
    aclOpExecutor* prepared[kMaxExecutors] = {nullptr};
    uint64_t sizes[kMaxExecutors] = {0};
    for (int run = 0; run < runs; ++run) {
      ACL_CHECK(aclnnAddGetWorkspaceSize(self, other, scalar, out, &sizes[run],
                                         &prepared[run]));
    }
    ACL_CHECK(aclrtRecordEvent(begin, stream));
    for (int run = 0; run < runs; ++run) {
      ACL_CHECK(aclnnAdd(workspace, sizes[run], prepared[run], stream));
    }
    ACL_CHECK(aclrtRecordEvent(end, stream));
    ACL_CHECK(aclrtSynchronizeStream(stream));
    float device_ms = 0.0f;
    ACL_CHECK(aclrtEventElapsedTime(&device_ms, begin, end));

    std::printf(
        "[PHASE] elems=%lld bytes=%lld ws=%llu first_us=%.3f second_us=%.3f "
        "device_us=%.3f\n",
        static_cast<long long>(counts[index]),
        static_cast<long long>(counts[index] * 4),
        static_cast<unsigned long long>(needed), first_ms * 1000.0 / runs,
        second_ms * 1000.0 / runs,
        static_cast<double>(device_ms) * 1000.0 / runs);

    (void)aclDestroyTensor(self);
    (void)aclDestroyTensor(other);
    (void)aclDestroyTensor(out);
  }

  (void)aclDestroyScalar(scalar);
  if (workspace != nullptr) {
    ACL_CHECK(aclrtFree(workspace));
  }
  for (int i = 0; i < 3; ++i) {
    ACL_CHECK(aclrtFree(buffer[i]));
  }
  ACL_CHECK(aclrtDestroyEvent(begin));
  ACL_CHECK(aclrtDestroyEvent(end));
  ACL_CHECK(aclrtDestroyStream(stream));
  std::printf("[RESULT] PASS\n");
  return ACL_SUCCESS;
}


### 8.5 视图与存储，以及主程序

`RunView` 在同一块存储上建立三个视图：连续、转置、右下角四分之一。三者的存储形状与首地址完全相同，**区别只在 `viewDims`、`strides` 与 `offset` 三项**。每个视图各参与一次 `aclnnAdd`，计算结果与主机侧按相同下标映射算出的期望值逐元素比较。

若算子不接受某个视图，第一段将返回非零码，程序把该返回码原样打印并将该行标记为 `SKIP`，不中止后续测量。

In [ ]:
%%writefile -a src_aclnn/acl_aclnn.cpp
// Builds three views over one storage buffer and checks each against a host
// computation. Nothing is copied: only the view shape, the strides and the
// offset change, while arguments 7 and 8 keep describing the same storage.
int RunView(int64_t n) {
  const int64_t half = n / 2;
  const int64_t count = n * n;
  std::vector<float> host_s(static_cast<size_t>(count));
  std::vector<float> host_b(static_cast<size_t>(count));
  std::vector<float> expected(static_cast<size_t>(count));
  std::vector<float> actual(static_cast<size_t>(count));
  FillMatrix(host_s.data(), n, n, 100);
  FillMatrix(host_b.data(), n, n, 97);

  aclrtStream stream = nullptr;
  ACL_CHECK(aclrtCreateStream(&stream));

  const int64_t full[kMatDims] = {n, n};
  const int64_t part[kMatDims] = {half, half};
  void* storage = nullptr;
  const size_t bytes = static_cast<size_t>(count) * sizeof(float);
  ACL_CHECK(aclrtMalloc(&storage, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMemcpy(storage, bytes, host_s.data(), bytes,
                        ACL_MEMCPY_HOST_TO_DEVICE));

  DeviceTensor other = {};
  DeviceTensor result = {};
  ACL_CHECK(CreateDeviceTensor(full, kMatDims, ACL_FLOAT, &other));
  ACL_CHECK(CreateDeviceTensor(full, kMatDims, ACL_FLOAT, &result));
  ACL_CHECK(aclrtMemcpy(other.addr, bytes, host_b.data(), bytes,
                        ACL_MEMCPY_HOST_TO_DEVICE));
  float one = kOneValue;
  aclScalar* scalar = aclCreateScalar(&one, ACL_FLOAT);
  if (scalar == nullptr) {
    std::fprintf(stderr,
                 "[ERR] api=aclCreateScalar code=- msg=returned null\n");
    return ACL_ERROR_INVALID_PARAM;
  }

  // Only `view` changes shape, strides and offset. `addend` and `target`
  // always describe the leading elements of their own buffers, so that each
  // case introduces exactly one variable.
  const char* names[3] = {"contiguous", "transposed", "quadrant"};
  const int64_t* view_shape[3] = {full, full, part};
  const int64_t strides[3][kMatDims] = {{n, 1}, {1, n}, {n, 1}};
  const int64_t offsets[3] = {0, 0, half * n + half};
  void* workspace = nullptr;
  uint64_t capacity = 0;
  bool all_passed = true;

  std::printf("[CFG] mode=view n=%lld\n", static_cast<long long>(n));
  for (int index = 0; index < 3; ++index) {
    const int64_t rows = view_shape[index][0];
    const int64_t cols = view_shape[index][1];
    const int64_t elements = rows * cols;
    for (int64_t i = 0; i < rows; ++i) {
      for (int64_t j = 0; j < cols; ++j) {
        const int64_t source =
            offsets[index] + i * strides[index][0] + j * strides[index][1];
        expected[static_cast<size_t>(i * cols + j)] =
            host_s[static_cast<size_t>(source)] +
            host_b[static_cast<size_t>(i * cols + j)];
      }
    }

    aclTensor* view =
        aclCreateTensor(view_shape[index], kMatDims, ACL_FLOAT, strides[index],
                        offsets[index], ACL_FORMAT_ND, full, kMatDims, storage);
    aclTensor* addend =
        MakeTensor(view_shape[index], kMatDims, ACL_FLOAT, other.addr);
    aclTensor* target =
        MakeTensor(view_shape[index], kMatDims, ACL_FLOAT, result.addr);
    if (view == nullptr || addend == nullptr || target == nullptr) {
      std::fprintf(stderr,
                   "[ERR] api=aclCreateTensor code=- msg=returned null\n");
      return ACL_ERROR_INVALID_PARAM;
    }

    uint64_t needed = 0;
    aclOpExecutor* executor = nullptr;
    const int status = aclnnAddGetWorkspaceSize(view, addend, scalar, target,
                                                &needed, &executor);
    if (status == ACL_SUCCESS) {
      ACL_CHECK(EnsureWorkspace(&workspace, &capacity, needed, stream));
      ACL_CHECK(aclnnAdd(workspace, needed, executor, stream));
      ACL_CHECK(aclrtSynchronizeStream(stream));
      const size_t taken = static_cast<size_t>(elements) * sizeof(float);
      ACL_CHECK(aclrtMemcpy(actual.data(), taken, result.addr, taken,
                            ACL_MEMCPY_DEVICE_TO_HOST));
      const double worst = MaxAbsDiff(actual.data(), expected.data(), elements);
      const bool ok = worst <= kRelTolerance;
      all_passed = all_passed && ok;
      std::printf(
          "[VIEW] case=%s dims=%lldx%lld strides=%lld,%lld offset=%lld "
          "status=0 err=%.4e check=%s\n",
          names[index], static_cast<long long>(rows),
          static_cast<long long>(cols),
          static_cast<long long>(strides[index][0]),
          static_cast<long long>(strides[index][1]),
          static_cast<long long>(offsets[index]), worst, ok ? "PASS" : "FAIL");
    } else {
      std::printf(
          "[VIEW] case=%s dims=%lldx%lld strides=%lld,%lld offset=%lld "
          "status=%d err=- check=SKIP\n",
          names[index], static_cast<long long>(rows),
          static_cast<long long>(cols),
          static_cast<long long>(strides[index][0]),
          static_cast<long long>(strides[index][1]),
          static_cast<long long>(offsets[index]), status);
    }
    (void)aclDestroyTensor(view);
    (void)aclDestroyTensor(addend);
    (void)aclDestroyTensor(target);
  }
  (void)aclDestroyScalar(scalar);
  if (workspace != nullptr) {
    ACL_CHECK(aclrtFree(workspace));
  }
  DestroyDeviceTensor(&result);
  DestroyDeviceTensor(&other);
  ACL_CHECK(aclrtFree(storage));
  ACL_CHECK(aclrtDestroyStream(stream));
  std::printf("[RESULT] %s\n", all_passed ? "PASS" : "FAIL");
  return all_passed ? ACL_SUCCESS : ACL_ERROR_INVALID_PARAM;
}

// Acquires the runtime resources that every mode needs.
int SetUp(aclrtContext* context) {
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(kDeviceId));
  ACL_CHECK(aclrtCreateContext(context, kDeviceId));
  ACL_CHECK(aclrtSetCurrentContext(*context));
  aclrtRunMode run_mode = ACL_HOST;
  ACL_CHECK(aclrtGetRunMode(&run_mode));
  std::printf("[ENV] soc=%s run_mode=%s\n", aclrtGetSocName(),
              (run_mode == ACL_HOST) ? "ACL_HOST" : "ACL_DEVICE");
  return ACL_SUCCESS;
}

// Releases them in the reverse order.
void TearDown(aclrtContext context) {
  if (context != nullptr) {
    (void)aclrtDestroyContext(context);
  }
  (void)aclrtResetDevice(kDeviceId);
  (void)aclFinalize();
}

int Dispatch(int argc, char** argv) {
  const char* mode = (argc > 1) ? argv[1] : "phase";
  aclrtContext context = nullptr;
  ACL_CHECK(SetUp(&context));

  int status = ACL_SUCCESS;
  if (std::strcmp(mode, "phase") == 0) {
    const int repeat = (argc > 2) ? std::atoi(argv[2]) : 10;
    status = RunPhase(repeat);
  } else if (std::strcmp(mode, "view") == 0) {
    const int64_t n = (argc > 2) ? std::atoll(argv[2]) : 64;
    status = RunView(n);
  } else {
    std::fprintf(stderr, "[ERR] api=main code=- msg=unknown mode %s\n", mode);
    status = ACL_ERROR_INVALID_PARAM;
  }

  TearDown(context);
  return status;
}

int main(int argc, char** argv) {
  return (Dispatch(argc, argv) == ACL_SUCCESS) ? 0 : 1;
}


## 9. 编译与运行

链接参数即 §6 检查得到的三个库：Runtime 库、算子公共数据类型库与 Math 类算子库（§4）。

In [ ]:
import os, subprocess

cmd = (
    ["g++", "src_aclnn/acl_aclnn.cpp", "-std=c++17", "-O2", "-Wall", "-Wextra"]
    + ["-I" + os.environ["ASCEND_HOME_PATH"] + "/include"]
    + ACL_LIBDIRS
    + ACL_LIBS
    + ["-o", "src_aclnn/acl_aclnn"]
)
print(" ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print((proc.stdout + proc.stderr).strip() or "✅ 编译通过")


两种模式各运行一次，参数取 §7.2 规定的值。两组测量合计通常在数秒内完成。


In [ ]:
import subprocess

runs = [
    ("phase", ["phase", "10"]),
    ("view", ["view", "512"]),
]
out = {}
for name, args in runs:
    proc = subprocess.run(
        ["./src_aclnn/acl_aclnn"] + args, capture_output=True, text=True
    )
    out[name] = proc.stdout
    tail = proc.stdout.strip().splitlines()[-1:] or ["(无输出)"]
    print(f"=== {name:6s} 退出码 {proc.returncode}  {tail[0]}")
    if proc.returncode != 0:
        print(proc.stderr.strip()[:800])


## 10. 解析输出

程序输出的每一行记录都采用 `[标签] 键=值 键=值 …` 的形式，因此可以用同一个函数解析。数值字段统一转换为 `float`，其余字段保留为字符串；`-` 表示该字段在这一行不适用。


In [ ]:
import re

NUMERIC = {
    "elems",
    "bytes",
    "ws",
    "first_us",
    "second_us",
    "device_us",
    "err",
    "status",
    "offset",
}


def parse(text, tag):
    "把带某个标签的所有记录行解析成字典列表。"
    rows = []
    for line in text.splitlines():
        if not line.startswith(tag + " "):
            continue
        item = {}
        for token in line[len(tag) + 1 :].split():
            if "=" not in token:
                continue
            key, value = token.split("=", 1)
            if key in NUMERIC and value != "-":
                item[key] = float(value)
            else:
                item[key] = value
        rows.append(item)
    return rows


phase = parse(out["phase"], "[PHASE]")
view = parse(out["view"], "[VIEW]")
print("解析到：phase %d 行，view %d 行" % (len(phase), len(view)))


## 11. 两段各自的主机侧耗时

下面这张表把三个数值并排列出：调用一次算子时，主机在第一段与第二段各花费多少时间，以及设备端完成这一次调用需要多少时间。

以下四项需要逐一核对：

- **第一段的耗时是否随数据量变化。** 表末给出了它在八档之间的取值范围与最大最小之比。若该比值接近 1，说明这部分耗时按调用次数计，而不按数据量计。
- **第一段与第二段哪一项耗时更长。** 两段所做的工作并不相同，第一段是入参校验与 Tiling，第二段是将任务提交到队列，两者的相对大小需要由实测确定。
- **`workspace` 一列。** §2.4 第 1 条指出 `workspaceSize` 可能为 0，这一列给出 `aclnnAdd` 在本机上是否需要临时工作区。
- **设备侧一列在小规模一端是否也保持平稳。** 若是，说明设备端同样存在一个与数据量无关的下限；它与主机两段之和相比哪一项更大，由下一格的图判断。

> **`workspace` 一列需要与 §6 探针的输出对照。** 若这一列八档全为 0，说明本实验所用的调用路径不申请临时工作区；此时省略 `if (workspaceSize > 0)` 判断会有什么后果，取决于探针打印的 `malloc_zero_ret`。


In [ ]:
print(
    "%12s %10s %14s %12s %12s %12s %10s"
    % (
        "元素数",
        "字节",
        "workspace",
        "第一段 (µs)",
        "第二段 (µs)",
        "设备侧 (µs)",
        "主机占比",
    )
)
print("-" * 88)
for row in phase:
    host = row["first_us"] + row["second_us"]
    share = host / (host + row["device_us"]) * 100.0
    print(
        "%12d %9.0fK %13.0fK %12.3f %12.3f %12.3f %9.1f%%"
        % (
            row["elems"],
            row["bytes"] / 1024,
            row["ws"] / 1024,
            row["first_us"],
            row["second_us"],
            row["device_us"],
            share,
        )
    )

first = [r["first_us"] for r in phase]
print()
print(
    "第一段耗时的范围：%.3f – %.3f µs，最大与最小之比 %.2f"
    % (min(first), max(first), max(first) / max(min(first), 1e-9))
)
print("同一区间内数据量变化了 %.0f 倍" % (phase[-1]["elems"] / phase[0]["elems"]))


下面把四条曲线画在同一张双对数坐标图上：第一段、第二段、主机两段合计、设备侧执行。**横轴是元素数，纵轴是每次调用的耗时。** 两条主机侧曲线若接近水平，说明这部分耗时与数据量无关；设备侧曲线的斜率若接近 1，说明设备端耗时与数据量成正比。

若两类曲线相交，图中会标出交点。

> **两项需要核对。** 第一，两条主机侧曲线在整个横轴上是否接近水平。若是，这部分耗时按调用次数计，而不按数据量计。第二，设备侧曲线在小规模一端是否也有一段水平区间。若有，说明设备端同样存在一个与数据量无关的启动下限。这个下限与主机侧固定开销相比哪一项更大，随机器而不同；上一格的代码会判断两条曲线是否相交并标出交点，交点是否存在本身就是本机的一项结果。
>
> **设备侧曲线离开水平区间之后，可以把每一档换算成等效带宽。** 一次 Add 读取两块、写入一块，共 $3 \times 4N$ 字节，除以设备侧耗时即得。请对离开水平区间之后的各档分别计算，列出元素数、传输量、设备侧耗时与等效带宽四列。**设备侧耗时开始随数据量增长的那个规模，就是把数据交由设备计算开始产生收益的规模。**


In [ ]:
%matplotlib inline

import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False

elems = [r["elems"] for r in phase]
host = [r["first_us"] + r["second_us"] for r in phase]

fig, ax = plt.subplots(figsize=(7.6, 4.4), dpi=120)
ax.plot(
    elems, [r["first_us"] for r in phase], "o-", label="phase 1 (host: check + tiling)"
)
ax.plot(elems, [r["second_us"] for r in phase], "s-", label="phase 2 (host: submit)")
ax.plot(elems, host, "^--", color="0.45", label="host total")
ax.plot(elems, [r["device_us"] for r in phase], "d-", label="device execution")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("elements")
ax.set_ylabel("time per call (us)")
ax.set_title("Two-phase interface: host cost and device time vs size")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=9)

# 找到设备侧耗时超过主机合计的第一个规模，在双对数坐标上线性插值
cross = None
for i in range(1, len(elems)):
    if phase[i - 1]["device_us"] < host[i - 1] and phase[i]["device_us"] >= host[i]:
        import math

        x0, x1 = math.log2(elems[i - 1]), math.log2(elems[i])
        d0 = math.log10(phase[i - 1]["device_us"]) - math.log10(host[i - 1])
        d1 = math.log10(phase[i]["device_us"]) - math.log10(host[i])
        cross = 2 ** (x0 + (x1 - x0) * (-d0) / (d1 - d0))
        break
if cross is not None:
    ax.axvline(cross, color="crimson", linestyle=":", linewidth=1.2)
    ax.annotate(
        "crossing at ~%.0f elements (%.1f KB)" % (cross, cross * 4 / 1024),
        xy=(cross, max(host)),
        xytext=(6, -12),
        textcoords="offset points",
        color="crimson",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

if cross is None:
    print("在本次扫描的规模范围内两条曲线没有相交，交点位于扫描区间之外。")
else:
    print("主机侧与设备侧两条曲线的交点：约 %.0f 个元素，即 %.1f KB" % (cross, cross * 4 / 1024))


## 12. 视图与存储：三个视图，一块存储

三个视图的**存储形状与首地址完全相同**，区别只在视图形状、strides 与 offset 三项。若三行全部通过，说明转置与切片确实可以在不复制数据的前提下表达。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 视图 | 视图形状 | strides | offset | 表达的内容 |
| --- | --- | --- | --- | --- |
| 连续 | $N \times N$ | $\{N,\,1\}$ | 0 | 存储本身 |
| 转置 | $N \times N$ | $\{1,\,N\}$ | 0 | $S^\mathsf{T}$，两个 stride 互换 |
| 四分之一 | $\tfrac N2 \times \tfrac N2$ | $\{N,\,1\}$ | $\tfrac N2 N + \tfrac N2$ | 右下角的子块，行距仍为 $N$ |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">视图</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">视图形状</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">strides</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">offset</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">表达的内容</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">连续</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$N \times N$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$\{N,\,1\}$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">0</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">存储本身</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">转置</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$N \times N$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$\{1,\,N\}$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">0</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$S^\mathsf{T}$，两个 stride 互换</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">四分之一</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$\tfrac N2 \times \tfrac N2$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$\{N,\,1\}$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$\tfrac N2 N + \tfrac N2$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">右下角的子块，行距仍为 $N$</td>
</tr>
</tbody>
</table>


In [ ]:
print(
    "%-12s %-12s %-12s %-10s %-8s %14s %-8s"
    % ("视图", "形状", "strides", "offset", "第一段", "最大绝对误差", "校验")
)
print("-" * 82)
for row in view:
    ok = row["check"]
    print(
        "%-12s %-12s %-12s %-10s %-8s %14s %-8s"
        % (
            row["case"],
            row["dims"],
            row["strides"],
            int(row["offset"]),
            "成功" if row["status"] == 0 else "返回 %d" % int(row["status"]),
            ("%.3e" % row["err"]) if "err" in row else "-",
            ok,
        )
    )

passed = [r for r in view if r["check"] == "PASS"]
skipped = [r for r in view if r["check"] == "SKIP"]
print()
print("三个视图中 %d 个通过校验，%d 个被算子拒绝。" % (len(passed), len(skipped)))
if skipped:
    print(
        "被拒绝的写法："
        + "、".join(r["case"] for r in skipped)
        + "。这说明该算子对非连续输入有额外要求，请查阅《算子库》中该算子的约束条件。"
    )
else:
    print("转置与切片都不需要复制数据，只需改变视图形状、strides 与 offset。")


## 13. 结果分析

本节给出的是结果的**判读方法与定性规律**。具体数值与设备型号、CANN 版本、编译选项以及运行时的负载都有关，请以自己运行得到的输出为准。

### 🎓 结论

**① 两段式接口的分工是主机准备与设备执行。** 第一段完成入参校验、输出 Shape 推导、Tiling 与 workspace 计算，全部在主机侧执行且是同步的；第二段把核函数下发到 Stream，是异步的。§11 的表分别列出了两段各自的主机耗时，以及设备端完成同一次调用所需的时间。

**② 第一段的耗时按调用次数计，与数据量无关。** `aclOpExecutor` 由第一段产生、被第二段消耗，因此**每一次调用都要重新执行第一段**，它不是一次性的初始化开销。§11 的表末给出了第一段耗时的取值范围与最大最小之比，而同一张表中数据量变化了四个数量级：**若该比值接近 1，即可判定这部分耗时按调用次数计。**

这一条支持 §5 的结论：用原子算子组合出一个复合运算，分成几步就要付出几倍的第一段耗时。对于这类固定开销，减少它的途径只有一条，即**减少调用次数**。

**③ 固定开销不只存在于主机侧。** 设备侧曲线在小规模一端同样有一段水平区间，说明启动一次设备任务本身也需要时间，而这部分耗时与数据量无关。它与主机两段之和相比哪一项更大，随机器而不同；第 36 单元格的代码会判断两条曲线是否相交并标出交点，交点是否存在本身就是本机的一项结果。

有一点需要注意：§11 测出的设备侧下限仍包含每次第二段的下发时间，因此它是**下发时间与启动时间之和**的下限，而不是核函数启动时间本身（§8.4）。

**④ workspace 可能始终为 0，但三条管理规则仍须遵守。** §11 的 `workspace` 一列给出 `aclnnAdd` 在本机上是否需要临时工作区。`aclrtMalloc` 申请 0 字节的行为以 §6 探针打印的 `malloc_zero_ret` 为准。三条规则中最容易被忽略的是第 3 条：同一条 Stream 上顺序执行的算子可以共用一块 workspace，**并发运行在不同 Stream 上的算子不能共用**。

**⑤ 视图与存储分开描述，使转置与切片不必复制数据。** §12 的三个视图指向同一块存储，只有 `viewDims`、`strides`、`offset` 三项不同，三行的误差应当为 0 或落在容差之内。这一设计在数据量较大时的价值更为明显：一次大矩阵的显式转置需要复制整块数据，而改变视图不需要任何数据传输。

### 一条方法论

**接口的耗时应当分项测量，而不是整体估算。** 本实验没有把一次算子调用作为整体计时，而是将它分为主机第一段、主机第二段、设备执行三项分别测量。只有分项测量，才能判断哪一项随数据量增长、哪一项不随，而这正是判断一项计算是否值得交由设备完成的依据。


## 14. 🔧 动手练习

> **提示**：两题都需要修改源码或编译命令。修改 `%%writefile` 单元格之后需要重新执行，由于 §8.1 为覆盖写、其后四个为追加写，**须从 8.1 开始按顺序重新执行**。

**1. 库文件拆分带来的链接错误。**

把 §9 编译命令中的 `-lopapi_math` 换成 `-lopapi`，记录报错信息；再删去这一项，记录报错信息。请说明两次报错分别发生在编译的哪个阶段、提示的是哪些符号。

结合 §4 的分类表回答两个问题：① CANN 8.5.0 将总库 `libopapi.so` 拆分为四个库的动机是什么？可以从一个应用只使用四类算子中的一类时的情形入手分析。② 若把本实验的 `aclnnAdd` 换成 `aclnnMatmul`，链接参数应当如何修改？

**2. 复用第一段产生的 executor。**

`RunPhase` 的第二遍循环先调用 `runs` 次第一段，得到 `runs` 个 executor，再连续下发 `runs` 次第二段。请把它改为**只调用一次第一段**，然后把同一个 executor 传给 `runs` 次第二段，记录实际结果：返回码是多少、程序是否异常退出、`device_us` 一列的数值有何变化。然后对照 §2.3 引用的官方说明，解释这种写法为什么不成立。

这种写法的后果在官方文档中没有明确规定，因此本题的结论只能来自实测与 §2.3 的那条要求，不能来自推测。请如实记录观察到的现象，包括程序看起来正常运行的情形：一种错误写法在某次运行中没有出现问题，并不能说明它是正确的。


## 15. 🤔 思考题

**1.** 第一段接口的名称是 `aclnnXxxGetWorkspaceSize`，但它完成的工作不止于获取 workspace 大小，还包括入参校验、Shape 推导与 Tiling。接口名称为什么只体现 workspace 这一项？请从调用者在两段之间必须完成的工作这一角度作答。若本机测得的 `workspaceSize` 始终为 0，这一名称是否更难理解？

**2.** `aclCreateTensor` 同时要求给出视图形状与存储形状。既然存储形状可以由存储的字节数与数据类型推算出来，为什么还要显式传入？可以从视图只覆盖存储一部分的情形入手分析。


## 16. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 要点 | 内容 |
| --- | --- |
| 两段式接口 | 第一段在主机侧同步执行，完成入参校验、Shape 推导、Tiling 与 workspace 计算；第二段异步下发核函数 |
| executor 不可复用 | 由第一段产生、被第二段消耗，每次调用都要重新执行第一段，因此**第一段的耗时按调用次数计** |
| workspace 三条规则 | 申请与释放前用 `if (workspaceSize > 0)` 判断；在同步之后释放；同一条 Stream 上的算子可共用，并发于不同 Stream 的算子不可共用 |
| workspace 可能为 0 | `workspaceSize` 为 0 是常见情形；该判断仍须保留，它使代码不依赖 `aclrtMalloc` 对 0 字节请求的处理方式 |
| `aclCreateTensor` 九个参数 | 1-2 为视图形状、4-5 为视图到存储的映射、7-8 为存储形状；**strides 以元素为单位** |
| 销毁顺序 | 先销毁描述符（`aclDestroyTensor`），再释放数据内存（`aclrtFree`）；次序颠倒后的错误通常在其后的某次分配上才显现 |
| 算子分类与库文件 | Math / NN / CV / Transformer 四类，`libopapi_math.so` 是四类共同的前提；`libopapi.so` 自 CANN 8.5.0 起废弃，不再设置为备选项 |
| 原子算子的组合 | 复合运算需要由多个 `aclnn` 算子组合而成，**分成几步就要付出几倍的第一段耗时**；换来的是算子库不必为每种组合各提供一个接口 |
| 固定开销分布在两侧 | 主机侧与设备侧**各有一个与数据量无关的下限**；两者哪一项更大随机器而不同，本机的结论取决于第 36 单元格是否找到交点 |
| 视图不需要复制数据 | 转置与切片只改变视图形状、strides 与 offset，存储形状与首地址保持不变 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内容</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两段式接口</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第一段在主机侧同步执行，完成入参校验、Shape 推导、Tiling 与 workspace 计算；第二段异步下发核函数</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">executor 不可复用</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由第一段产生、被第二段消耗，每次调用都要重新执行第一段，因此<strong>第一段的耗时按调用次数计</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">workspace 三条规则</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">申请与释放前用 <code>if (workspaceSize &gt; 0)</code> 判断；在同步之后释放；同一条 Stream 上的算子可共用，并发于不同 Stream 的算子不可共用</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">workspace 可能为 0</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>workspaceSize</code> 为 0 是常见情形；该判断仍须保留，它使代码不依赖 <code>aclrtMalloc</code> 对 0 字节请求的处理方式</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclCreateTensor</code> 九个参数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1-2 为视图形状、4-5 为视图到存储的映射、7-8 为存储形状；<strong>strides 以元素为单位</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">销毁顺序</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">先销毁描述符（<code>aclDestroyTensor</code>），再释放数据内存（<code>aclrtFree</code>）；次序颠倒后的错误通常在其后的某次分配上才显现</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子分类与库文件</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Math / NN / CV / Transformer 四类，<code>libopapi_math.so</code> 是四类共同的前提；<code>libopapi.so</code> 自 CANN 8.5.0 起废弃，不再设置为备选项</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">原子算子的组合</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">复合运算需要由多个 <code>aclnn</code> 算子组合而成，<strong>分成几步就要付出几倍的第一段耗时</strong>；换来的是算子库不必为每种组合各提供一个接口</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">固定开销分布在两侧</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">主机侧与设备侧<strong>各有一个与数据量无关的下限</strong>；两者哪一项更大随机器而不同，本机的结论取决于第 36 单元格是否找到交点</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">视图不需要复制数据</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">转置与切片只改变视图形状、strides 与 offset，存储形状与首地址保持不变</td>
</tr>
</tbody>
</table>

### 一条贯穿本章的原则

> **每一次把计算任务交给设备，都要付出一份与工作量无关的固定开销。这份开销决定了一项任务小到何种规模就不再值得交给设备完成。**

本实验把这条原则用于算子调用，并且在此基础上多出一层结论：**固定开销不只存在于主机侧。** §11 的两条曲线在小规模一端都保持水平，主机侧每次调用需要重新执行第一段，设备侧每次任务需要付出一份启动开销。两者哪一项更大随机器而不同，但都不随数据量增长，因此减少它们的途径是同一条，即减少调用次数。

判读这类数据时还有一点需要记住：**一个性能数值偏大，并不意味着问题一定出在预期的那一侧。** §11 把一次调用的耗时分为三项分别测量，正是为了使时间的实际分布有据可依，而不依赖推断。
